# ModHav Synthetic Dataset v2

In [6]:
# ── 0. Imports ─────────────────────────────────────────────────────────────────
from config.config import *
from config.experiments import ExperimentConfig

import src.data.saving as saving
import src.utils.visuals as visuals

from src.circuit.candidates import OptunaCandidate
from src.kernel.quantum_kernel import quantum_kernel_matrix
from src.kernel.metrics import geometric_difference
from typing import Callable
from sklearn.metrics.pairwise import rbf_kernel
import optuna
from src.data.saving import save_parameter_train
from src.optimization.objectives import objective_optuna_generator_HD

from src.data.preprocessing import preprocess_synthetic
from src.data.synthetic import Havlicek_synthetic_dataset, ModHav_synthetic_dataset
from src.data.saving import candidate_exists, svm_exists, load_parameter_train, load_svm_results
from src.data.subsets import subset_random, subset_nystrom_global, subset_nystrom_stratified
from src.kernel.quantum_kernel import quantum_kernel_matrix_cross
from src.optimization.pipeline import get_candidate, get_svm_results
from src.optimization.classical_base import classical_baseline
from src.utils.visuals import *

In [7]:
# –– NEW STAGE: TRAINING OF ANGLES ––––––––––––––––––––––––––––––––––––––––––––––

def kernel_target_alignment(K: np.ndarray, y: np.ndarray) -> float:
    """
    How well kernel K's similarity structure matches the class labels y.
    Higher = better alignment with same-class vs different-class pairs —
    a cheap proxy for SVM accuracy that doesn't require fitting an SVM.
    """
    y = np.asarray(y).reshape(-1, 1)
    yyT = y @ y.T
    return float(np.sum(K * yyT) / (np.linalg.norm(K) * np.linalg.norm(yyT)))

def objective_optuna_angles_generator(
    X: np.ndarray, y: np.ndarray, gates: list
) -> Callable[[optuna.Trial], float]:
    """
    Gate structure is fixed (from stage 1). Only angles are optimized,
    against kernel-target alignment on the training subset.
    Args:
        X: (n, d) subset of training points
        y: labels for X
        gates: fixed gate list, one per gene
    """
    def objective(trial):
        angles = [trial.suggest_float(f"angle_{i}", 0, 2 * np.pi) for i in range(len(gates))]
        candidate = OptunaCandidate(N_QUBITS, N_LAYERS, gates, angles)
        K_quantum = quantum_kernel_matrix(X, candidate)
        return kernel_target_alignment(K_quantum, y)
    return objective

def train_candidate_optuna_angles(
    subset_type: str,
    X_subset: np.ndarray, y_subset: np.ndarray,
    gates: list,
    n_qubits: int = N_QUBITS, n_trials: int = N_TRIALS, n_layers: int = N_LAYERS,
) -> tuple[float, OptunaCandidate, np.ndarray, np.ndarray]:
    """Stage 2: refine angles for a fixed gate structure from stage 1."""
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_optuna_angles_generator(X_subset, y_subset, gates), n_trials=n_trials)

    best_angles = [study.best_params[f"angle_{i}"] for i in range(n_qubits * n_layers)]
    best_candidate = OptunaCandidate(n_qubits, n_layers, gates, best_angles)

    K_quantum_best = quantum_kernel_matrix(X_subset, best_candidate)
    K_classical = rbf_kernel(X_subset, gamma=1.0 / (n_qubits * X_subset.var()))

    save_parameter_train(subset_type, best_candidate, study.best_value, K_quantum_best, K_classical, X_subset, y_subset)
    return study.best_value, best_candidate, K_quantum_best, K_classical

In [8]:
# ── 1. Data ────────────────────────────────────────────────────────────────
X, y = ModHav_synthetic_dataset(seed=45)
X_train, X_test, y_train, y_test = preprocess_synthetic(X, y, N_QUBITS)

RESULTS_DIR = 'Results/ModHav Synthetic v2'
saving.RESULTS_DIR = RESULTS_DIR
visuals.RESULTS_DIR = RESULTS_DIR

In [9]:
# ── Stage 1: search gate structure (Haar-sampled angles, as before) ────────
stage1_config = ExperimentConfig(
    name             = 'stage1_optuna_haar',
    subset_method    = 'none',
    optimizer        = 'optuna',
    variant          = 'haar',
    optimizer_kwargs = {'n_trials': 100},
)
candidate1, K_train1, K_classical1, X_sub, y_sub = get_candidate(stage1_config, X_train, y_train)
gates = [gene.gate for gene in candidate1.genes]

# ── Stage 2: refine angles for that fixed gate structure ───────────────────
stage2_name = 'stage2_optuna_angles'
if candidate_exists(stage2_name):
    data = load_parameter_train(stage2_name)
    candidate2, K_train2 = data['candidate'], data['K_quantum']
else:
    _, candidate2, K_train2, _ = train_candidate_optuna_angles(
        stage2_name, X_sub, y_sub, gates, n_trials=100
    )

[I 2026-08-20 14:45:52,422] A new study created in memory with name: no-name-95081577-114b-4776-8be5-632d131a994f
[I 2026-08-20 14:45:55,953] Trial 0 finished with value: 7.935707093913891 and parameters: {'gate_0': 'H', 'gate_1': 'CNOT', 'gate_2': 'I', 'gate_3': 'CNOT', 'gate_4': 'H', 'gate_5': 'I', 'gate_6': 'RZ', 'gate_7': 'CNOT', 'gate_8': 'RY', 'gate_9': 'RY', 'gate_10': 'RX', 'gate_11': 'RY', 'gate_12': 'CNOT', 'gate_13': 'H', 'gate_14': 'RX', 'gate_15': 'RX'}. Best is trial 0 with value: 7.935707093913891.
[I 2026-08-20 14:46:00,143] Trial 1 finished with value: 7.23131231702982 and parameters: {'gate_0': 'I', 'gate_1': 'RY', 'gate_2': 'H', 'gate_3': 'RX', 'gate_4': 'CNOT', 'gate_5': 'H', 'gate_6': 'RZ', 'gate_7': 'CNOT', 'gate_8': 'RX', 'gate_9': 'RZ', 'gate_10': 'CNOT', 'gate_11': 'RX', 'gate_12': 'RY', 'gate_13': 'RY', 'gate_14': 'H', 'gate_15': 'H'}. Best is trial 0 with value: 7.935707093913891.
[I 2026-08-20 14:46:04,020] Trial 2 finished with value: 9.451803653947277 and 

Saved candidate  → Results/ModHav Synthetic v2/stage1_optuna_haar/candidate.pkl


[I 2026-08-20 14:50:24,950] Trial 0 finished with value: 0.006179745624200018 and parameters: {'angle_0': 5.777418582122875, 'angle_1': 0.12437768649232063, 'angle_2': 5.016762967297089, 'angle_3': 2.3646526461280217, 'angle_4': 0.567229481630997, 'angle_5': 5.763042571721968, 'angle_6': 5.249955201497318, 'angle_7': 4.95196452136701, 'angle_8': 3.8452658429410427, 'angle_9': 4.225988992375516, 'angle_10': 4.989601741090023, 'angle_11': 0.7967531427354378, 'angle_12': 3.638436932677513, 'angle_13': 5.258423842730932, 'angle_14': 2.3809472569417154, 'angle_15': 1.905253446296535}. Best is trial 0 with value: 0.006179745624200018.
[I 2026-08-20 14:50:27,278] Trial 1 finished with value: 0.01885439163255159 and parameters: {'angle_0': 2.091583429923377, 'angle_1': 2.6964124258033957, 'angle_2': 0.6463432241801752, 'angle_3': 3.8299111023149406, 'angle_4': 3.297997518259608, 'angle_5': 5.089685909701727, 'angle_6': 3.0064962239018937, 'angle_7': 0.6061560906319263, 'angle_8': 5.96020236555

Saved candidate  → Results/ModHav Synthetic v2/stage2_optuna_angles/candidate.pkl


In [10]:
stage2_config = ExperimentConfig(
    name          = stage2_name,
    optimizer     = 'optuna',
    subset_method = 'none',
)

K_test = None
if not svm_exists(stage2_config.name):
    print("Computing cross-kernel (test × subset)…")
    K_test = quantum_kernel_matrix_cross(X_test, X_sub, candidate2)

results = get_svm_results(stage2_config, K_train2, y_sub, K_test, y_test)
print(f"Stage 2 test accuracy: {results['accuracy']:.3f}")

Computing cross-kernel (test × subset)…
Saved SVM        → Results/ModHav Synthetic v2/stage2_optuna_angles/svm.pkl

REPORT: stage2_optuna_angles | 100 trials
Best C            : 0.1
Best CV accuracy  : 0.9500
Test accuracy     : 1.0000
              precision    recall  f1-score   support

          -1       1.00      1.00      1.00         6
           1       1.00      1.00      1.00         6

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12

Stage 2 test accuracy: 1.000
